# Bước 6: Model evaluation – đánh giá mô hình

Chấm mô hình trên tập test, tập dữ liệu mà nó **chưa từng thấy**, để ước lượng nó sẽ làm gì với khách hàng mới.

Notebook này đọc kết quả do pipeline tạo ra. **Chạy `kedro run` trước** (một lần), sau đó mới chạy notebook này.

In [ ]:
import importlib.util
import logging
import sys
import warnings
from pathlib import Path

# Kiểm tra kernel đã có đủ thư viện chưa, để báo lỗi dễ hiểu nếu thiếu
required = ["kedro", "kedro_datasets", "pandas", "pyarrow", "scipy", "sklearn", "matplotlib"]
missing = [m for m in required if importlib.util.find_spec(m) is None]
if missing:
    raise ModuleNotFoundError(
        f"Kernel này ({sys.executable}) thiếu: {missing}.\n"
        "Chạy trong terminal rồi khởi động lại kernel:\n"
        f'  "{sys.executable}" -m pip install -r requirements.txt'
    )

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from kedro.framework.session import KedroSession
from kedro.framework.startup import bootstrap_project


def find_project_root() -> Path:
    # Tìm thư mục gốc của dự án Kedro, dù notebook được mở từ đâu
    here = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "x")).resolve().parent
    for folder in (here, *here.parents, Path.cwd(), *Path.cwd().parents):
        if (folder / "conf").is_dir() and (folder / "pyproject.toml").exists():
            return folder
    for pyproject in Path.cwd().glob("*/pyproject.toml"):  # mở từ thư mục cha
        if "[tool.kedro]" in pyproject.read_text(encoding="utf-8"):
            return pyproject.parent
    raise FileNotFoundError("Không tìm thấy dự án Kedro (thư mục có conf/ và pyproject.toml)")


project_path = find_project_root()
bootstrap_project(project_path)  # thêm thư mục src vào đường dẫn import
with KedroSession.create(project_path=project_path) as session:
    context = session.load_context()
    catalog = context.catalog  # dữ liệu khai báo trong conf/base/catalog.yml
    params = context.params    # tham số trong conf/base/parameters*.yml

for name in ("kedro", "netflix"):  # bớt log của Kedro trong notebook
    logging.getLogger(name).setLevel(logging.WARNING)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("Python đang dùng:", sys.executable)
print("Thư mục dự án:", project_path)

In [ ]:
import io

from IPython.display import Image, display


def show(fig, dpi=110):
    # Hiển thị một Figure trong notebook
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi)
    display(Image(buffer.getvalue()))

In [ ]:
from sklearn.metrics import precision_score, recall_score

ready = catalog.exists("classifier") and catalog.exists("metrics")
if not ready:
    raise RuntimeError("Chưa có mô hình hay metrics. Chạy `kedro run` trong terminal trước, rồi chạy lại notebook này.")

classifier = catalog.load("classifier")
X_test = catalog.load("X_test")
y_test = catalog.load("y_test")["churned"]
metrics = catalog.load("metrics")
figures = project_path / "data" / "08_reporting" / "evaluation" / "figures"
print("Mô hình:", type(classifier.named_steps["model"]).__name__, "| số khách test:", len(X_test))

## 6.1 Điểm tổng quát
- `baseline_accuracy`: độ chính xác nếu **luôn đoán lớp phổ biến nhất**. Mô hình chỉ có ích khi `accuracy` vượt rõ mức này.
- `precision`: trong số khách mô hình báo "sẽ rời", bao nhiêu % đúng là rời.
- `recall`: trong số khách thật sự rời, mô hình bắt được bao nhiêu %.
- `roc_auc`: khả năng xếp khách rời cao hơn khách ở lại (0,5 là tung đồng xu, 1,0 là hoàn hảo).

In [ ]:
pd.Series(metrics).to_frame("giá trị")

## 6.2 Mô hình sai ở đâu? (ma trận nhầm lẫn)
Bốn ô: đoán đúng "ở lại", đoán nhầm "rời" (báo động giả), bỏ sót khách sắp rời, đoán đúng "rời".
Với bài toán giữ khách, **bỏ sót** (ô dưới trái) thường tốn kém hơn báo động giả, vì mất khách thật.

In [ ]:
display(Image(filename=str(figures / "confusion_matrix.png")))

## 6.3 Đường ROC
Đường càng uốn về góc trên trái càng tốt. Đường chéo là đoán ngẫu nhiên.

In [ ]:
display(Image(filename=str(figures / "roc_curve.png")))

## 6.4 Điểm có đồng đều giữa các nhóm khách không?
Điểm tổng thể tốt có thể che một nhóm bị mô hình đoán tệ. Bảng dưới tách theo `subscription_type`
(đổi cột trong `conf/base/parameters_evaluation.yml`).

In [ ]:
catalog.load("metrics_by_segment")

## 6.5 Mô hình dựa vào cột nào? (permutation importance)
Xáo trộn ngẫu nhiên một cột rồi xem điểm tụt bao nhiêu. Tụt nhiều nghĩa là mô hình **dựa nhiều** vào cột đó.
Đối chiếu với bước 2: các cột mạnh ở đó có đứng đầu ở đây không?

In [ ]:
display(Image(filename=str(figures / "feature_importance.png")))
catalog.load("feature_importance")

## 6.6 Chọn ngưỡng quyết định
Mô hình cho ra **xác suất rời**. Ngưỡng 0,5 (`decision.threshold` trong `parameters.yml`) là mặc định, không phải chân lý.
- Hạ ngưỡng: bắt được nhiều khách sắp rời hơn (recall tăng) nhưng báo động giả nhiều hơn (precision giảm).
- Nâng ngưỡng: ngược lại.

Chọn ngưỡng là **quyết định kinh doanh**: một cuộc gọi giữ chân tốn bao nhiêu, mất một khách tốn bao nhiêu?

In [ ]:
proba = classifier.predict_proba(X_test)[:, 1]
rows = []
for threshold in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (proba >= threshold).astype(int)
    rows.append(
        {
            "ngưỡng": threshold,
            "số khách bị báo rời": int(pred.sum()),
            "precision": round(precision_score(y_test, pred, zero_division=0), 3),
            "recall": round(recall_score(y_test, pred, zero_division=0), 3),
        }
    )
pd.DataFrame(rows)

## 6.7 Những khách mô hình đoán sai nặng nhất
Nhìn trực tiếp các trường hợp sai thường gợi ý ra feature còn thiếu.
Dưới đây là khách **thật sự rời** nhưng mô hình cho xác suất rời **thấp nhất**.

In [ ]:
errors = X_test.assign(churn_probability=proba.round(3), churned=y_test.to_numpy())
errors[errors["churned"] == 1].sort_values("churn_probability").head(10)

## 6.8 Bảng tham khảo: cả 5 mô hình trên tập test
Bảng này chấm **từng mô hình ứng viên** trên tập test. Mô hình dòng `selected = True` là mô hình đã được chọn ở bước 5 (nhờ kiểm tra chéo trên tập train); `cv_rank` là thứ hạng lúc đó.

> **Chỉ để báo cáo, không dùng để chọn lại.** Nếu thấy một mô hình khác có điểm test cao hơn chút, đừng đổi sang nó: lúc đó tập test bị dùng để *quyết định*, và điểm số báo cáo sẽ đẹp hơn thực tế. Chênh lệch nhỏ như vậy thường chỉ là may rủi.
> Các dòng được xếp theo `cv_rank` (thứ tự lúc chọn), không xếp theo điểm test, cũng vì lý do này.

In [ ]:
comparison_test = catalog.load("model_comparison_test")
comparison_test

Cách đọc nhanh:
- Tìm cột `recall`: mô hình nào bắt được nhiều khách sắp rời nhất? Nhớ rằng recall còn phụ thuộc vào ngưỡng quyết định (mục 6.6).
- So `roc_auc` giữa các dòng: nếu các mô hình đứng đầu chênh nhau rất ít thì chọn cái **đơn giản hơn hoặc nhanh hơn** cũng hợp lý (xem `fit_seconds` trong `model_comparison.csv`).

## Ghi chú của bạn
Viết vài dòng bằng lời của bạn: bạn học được gì, điều gì bất ngờ, bạn sẽ thử gì tiếp?

-
-
-